In [ ]:
from pyspark.sql import DataFrame, functions as F

from transforms.backtest import paired_brier_difference, split_by_date
from transforms.calibration import (
    to_long_format,
    fit_all_calibration_curves,
    calibrate_matches,
    evaluate_calibration,
)

SILVER_TABLE = "stryktipset.silver.matches"

# Everything before this trains; this day and after is scored.
# ~80/20 on the played history, and a date rather than a percentage so the
# boundary lands on a season edge instead of slicing a draw in half.
CUTOFF = "2023-01-01"

In [ ]:
def describe(name: str, df: DataFrame) -> None:
    """Row count and date range, so the split is visible rather than assumed."""
    row = df.agg(
        F.count("*").alias("n"),
        F.min("match_start").alias("first"),
        F.max("match_start").alias("last"),
    ).first()
    print(f"{name:<6} {row['n']:>6} matches   {str(row['first'])[:10]} .. {str(row['last'])[:10]}")

In [ ]:
def main():
    """Fit the calibration curves on the early years only, then score the
    later ones -- matches the curves have never seen.

    This does NOT touch the production path. 05 still fits on every match,
    which is right: the model scoring next week's coupon should have seen
    as much data as possible. The model built here is thrown away; it
    exists only to produce one honest number.

    The comparison is raw streck vs calibrated streck on the test period.
    Lower Brier is better. If calibrated wins here, the correction is real
    -- unlike 05's in-sample scores, where it wins by construction.

    Elo is deliberately not scored here. elo_expected_score is an expected
    score on Elo's chess scale (a draw counts as half a win), not a 1X2
    probability, so it cannot go into a Brier score against 1/X/2 without
    a draw model first.
    """
    matches = spark.table(SILVER_TABLE)
    train, test = split_by_date(matches, CUTOFF)

    print(f"cutoff {CUTOFF}")
    describe("train", train)
    describe("test", test)
    print()

    models = fit_all_calibration_curves(to_long_format(train))
    scored = calibrate_matches(test, models)

    metrics = evaluate_calibration(scored)

    raw = metrics["brier_raw_pooled"]
    cal = metrics["brier_calibrated_pooled"]
    delta = raw - cal
    print()
    print(f"pooled raw        {raw:.4f}")
    print(f"pooled calibrated {cal:.4f}")
    print(f"difference        {delta:+.4f}  ({'calibrated better' if delta > 0 else 'raw better'})")

    # Is that difference real, or noise? Paired, because both numbers
    # describe the same matches -- see paired_brier_difference.
    stats = paired_brier_difference(scored)
    print()
    print(f"paired mean difference {stats['mean_difference']:+.5f}")
    print(f"standard error         {stats['standard_error']:.5f}")
    print(f"95% interval           {stats['ci_low']:+.5f} .. {stats['ci_high']:+.5f}")
    print(f"t                      {stats['t_statistic']:.2f}  (n = {stats['n']})")
    print()
    if stats["ci_low"] > 0:
        print("The interval excludes zero: the improvement is real at this sample size.")
    elif stats["ci_high"] < 0:
        print("The interval is entirely below zero: calibration is making things WORSE.")
    else:
        print("The interval straddles zero: this data cannot tell them apart.")

    dbutils.notebook.exit(
        f"cutoff={CUTOFF} train={train.count()} test={test.count()} "
        f"raw={raw:.4f} calibrated={cal:.4f} delta={delta:+.4f} "
        f"se={stats['standard_error']:.5f} t={stats['t_statistic']:.2f} "
        f"ci={stats['ci_low']:+.5f}..{stats['ci_high']:+.5f}"
    )

main()